In [279]:
import torch

In [280]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [281]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using {device} device")

Using cpu device


## Inheritance

In [282]:
# parent class

class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} says hello!"

# child class
class Dog(Animal):
    def bark(self):
        return f"{self.name} says woof!"

class Cat(Animal):
    def meow(self):
        return f"{self.name} says meow!"


# create an instance of the Dog class
dog1 = Dog("Buddy")
cat1 = Cat("Whiskers")

print(dog1.bark())
print(dog1.speak())
print(dog1.name)

print("---")

print(cat1.meow())
print(cat1.speak())
print(cat1.name)

Buddy says woof!
Buddy says hello!
Buddy
---
Whiskers says meow!
Whiskers says hello!
Whiskers


## `super()` - is used to access methods and attributes from the parent class
 - `__init__()` the child class is calling the parent class constructor
 - Independent variables/ Predictors / Features (X)
 - Dependent variables / Response / Label / Target (y)
 - `y=mx+b` b is the bias variable

In [283]:
class LinearRegression(nn.Module):
    def __init__(self, input_size=1, output_size=1):
        super(LinearRegression, self).__init__()

        # 1 input feature, 1 output feature
        self.linear = nn.Linear(input_size, output_size) # y = wx + b

    def forward(self, x):
        return self.linear(x)

In [284]:
# create an instance of the LinearRegression class
model = LinearRegression().to(device)
print(model)

for param in model.parameters():
    print(param.data)

LinearRegression(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)
tensor([[0.5187]])
tensor([0.8796])


In [285]:
print(list(model.parameters()))

[Parameter containing:
tensor([[0.5187]], requires_grad=True), Parameter containing:
tensor([0.8796], requires_grad=True)]


## Define out data

- normalization makes the data centered around the mean (sigma) falls between -1.0 and 1.0

In [286]:
weight = torch.tensor([[140], [155], [159], [179], [192], [200], [212]]).float()
height = torch.tensor([[60],  [62],  [67],  [70],  [71],  [72],  [75]]).float()

w_mean = weight.mean() # mean is the average value of the data
w_std = weight.std() # standard deviation is a measure of how spread out the data is
print(f"Weight mean: {w_mean:.2f}, std: {w_std:.2f}")

h_mean = height.mean()
h_std = height.std()
print(f"Height mean: {h_mean:.2f}, std: {h_std:.2f}")

# normalize the data
weight_x = (weight - weight.mean()) / weight.std()
height_y = (height - height.mean()) / height.std()


print("--- Normalized data ---")
print(weight_x)
print(height_y)

print(weight.shape)
print(height.shape)

Weight mean: 176.71, std: 26.33
Height mean: 68.14, std: 5.46
--- Normalized data ---
tensor([[-1.3944],
        [-0.8247],
        [-0.6728],
        [ 0.0868],
        [ 0.5806],
        [ 0.8844],
        [ 1.3402]])
tensor([[-1.4914],
        [-1.1251],
        [-0.2093],
        [ 0.3401],
        [ 0.5233],
        [ 0.7065],
        [ 1.2559]])
torch.Size([7, 1])
torch.Size([7, 1])


## Loss and Optimizer

In [287]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

## Training Loop

In [288]:
epochs = 1000

for epoch in range(epochs):
    model.train() # set the model to training mode

    # forward pass
    y_pred = model(weight_x.float()) # make predictions using the model

    # calculate loss
    loss = loss_fn(y_pred, height_y.float()) # calculate the loss between the predicted and actual values

    # backward pass
    loss.backward() # calculate the gradients dw and db

    # update weights
    optimizer.step() # w = w - lr * dw

    # zero gradients
    optimizer.zero_grad() # set the gradients to zero after updating weights

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

Epoch 0: Loss = 1.0034
Epoch 100: Loss = 0.0780


Epoch 200: Loss = 0.0594
Epoch 300: Loss = 0.0591
Epoch 400: Loss = 0.0590
Epoch 500: Loss = 0.0590
Epoch 600: Loss = 0.0590
Epoch 700: Loss = 0.0590
Epoch 800: Loss = 0.0590
Epoch 900: Loss = 0.0590


## Make Prediction

In [289]:
for param in model.parameters():
    print(param.data)

tensor([[0.9649]])
tensor([-8.0486e-07])


`height = 0.9649 * weight -8.3498e-07`

In [298]:
(180-w_mean) / w_std

tensor(0.1248)

In [301]:
test = torch.tensor([[(180-w_mean) / w_std]]) # normalize the test data
predicted_height = model(test.float())
print(f"Predicted height for weight 180: {predicted_height.item():.2f}")

Predicted height for weight 180: 0.12


In [291]:
(weight_x * w_std) + w_mean

tensor([[140.],
        [155.],
        [159.],
        [179.],
        [192.],
        [200.],
        [212.]])

In [304]:
height_y >= 0.12

tensor([[False],
        [False],
        [False],
        [ True],
        [ True],
        [ True],
        [ True]])

In [293]:
x = torch.tensor([
    [140.],
    [155.],
    [159.]
])

mean = x.mean()
std = x.std()

normalized_x = (x - mean) / std
print(normalized_x)

tensor([[-1.1314],
        [ 0.3661],
        [ 0.7654]])


In [294]:
# reversing the normalization
reversed_x = (normalized_x * std) + mean
print(reversed_x)

tensor([[140.],
        [155.],
        [159.]])
